# Notebook 2: Embeddings & Vector Space
**LLM Fundamentals Demo Series — Agentic AI Bootcamp**

This notebook demonstrates how tokens and words become **dense numerical vectors** (embeddings),
and how semantic meaning emerges in that vector space.

We use **DistilBERT** (66M parameters, publicly available on HuggingFace) as our model.

Topics covered:
1. What an embedding is (token → vector)
2. Cosine similarity between words
3. 2-D PCA projection of semantic word groups
4. Sentence embeddings and semantic search
5. Visualizing how context changes embeddings (polysemy)

In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel

# Load DistilBERT — downloads ~250 MB on first run
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print(f'Model loaded: {MODEL_NAME}')
print(f'Hidden size (embedding dim): {model.config.hidden_size}')
print(f'Number of layers           : {model.config.num_hidden_layers}')
print(f'Number of attention heads  : {model.config.num_attention_heads}')

## Helper: Get Contextual Embeddings
DistilBERT produces **contextual** embeddings: the vector for "bank" in
*river bank* differs from "bank" in *savings bank*.

In [ ]:
def get_embedding(text: str, strategy: str = 'cls') -> np.ndarray:
    """
    Returns a sentence embedding for `text`.
    strategy='cls'  : use the [CLS] token embedding (good for classification)
    strategy='mean' : mean-pool all non-padding token embeddings (better for similarity)
    """
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**enc)
    hidden = out.last_hidden_state  # (1, seq_len, 768)

    if strategy == 'cls':
        return hidden[0, 0, :].numpy()
    else:  # mean pool
        mask = enc['attention_mask'][0].unsqueeze(-1).float()
        summed = (hidden[0] * mask).sum(dim=0)
        return (summed / mask.sum()).numpy()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Quick sanity check
emb = get_embedding("Hello world", strategy='mean')
print(f'Embedding shape: {emb.shape}')   # should be (768,)

## 1. Cosine Similarity — How Semantically Close Are These Words?

In [ ]:
word_pairs = [
    ('king',   'queen'),
    ('king',   'man'),
    ('dog',    'cat'),
    ('dog',    'airplane'),
    ('happy',  'joyful'),
    ('happy',  'sad'),
    ('Python', 'Java'),
    ('Python', 'banana'),
]

print(f'{"Word A":<12} {"Word B":<12} {"Cosine Sim":>12}')
print('-' * 38)
for w1, w2 in word_pairs:
    e1 = get_embedding(w1, 'mean')
    e2 = get_embedding(w2, 'mean')
    sim = cosine_similarity(e1, e2)
    bar = '█' * int(sim * 20)
    print(f'{w1:<12} {w2:<12} {sim:>8.4f}  {bar}')

## 2. Similarity Heatmap Across a Word Set

In [ ]:
words = [
    # Animals
    'dog', 'cat', 'wolf', 'lion',
    # Royalty
    'king', 'queen', 'prince', 'duke',
    # Technology
    'Python', 'Java', 'software', 'algorithm',
    # Emotions
    'happy', 'sad', 'angry', 'excited',
]

embeddings = np.stack([get_embedding(w, 'mean') for w in words])

# Compute full similarity matrix
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normed = embeddings / norms
sim_matrix = normed @ normed.T

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix,
            xticklabels=words, yticklabels=words,
            annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, linewidths=0.5, ax=ax)
ax.set_title('Cosine Similarity Matrix — DistilBERT Embeddings', fontsize=13)
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

## 3. 2-D PCA Projection of Semantic Word Groups
We project the 768-dimensional embeddings down to 2D to visualize clusters.

In [ ]:
groups = {
    'Royalty':    ['king', 'queen', 'prince', 'princess', 'duke', 'throne'],
    'Animals':    ['dog', 'cat', 'wolf', 'lion', 'tiger', 'horse'],
    'Tech':       ['Python', 'Java', 'algorithm', 'software', 'computer', 'neural'],
    'Emotions':   ['happy', 'sad', 'angry', 'fearful', 'joyful', 'anxious'],
    'Geography':  ['mountain', 'river', 'ocean', 'desert', 'forest', 'valley'],
}

all_words, all_labels, all_embs = [], [], []
for label, words in groups.items():
    for w in words:
        all_words.append(w)
        all_labels.append(label)
        all_embs.append(get_embedding(w, 'mean'))

emb_matrix = np.stack(all_embs)

# PCA to 2D
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(emb_matrix)

# Plot
colors = plt.cm.tab10(np.linspace(0, 1, len(groups)))
color_map = {lbl: colors[i] for i, lbl in enumerate(groups)}

fig, ax = plt.subplots(figsize=(11, 8))
for word, label, (x, y) in zip(all_words, all_labels, coords_2d):
    ax.scatter(x, y, color=color_map[label], s=120, alpha=0.85, edgecolors='black', linewidth=0.4)
    ax.annotate(word, (x, y), textcoords='offset points', xytext=(6, 4), fontsize=9)

# Legend
handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=color_map[lbl], markersize=11, label=lbl)
           for lbl in groups]
ax.legend(handles=handles, title='Category', fontsize=10)
ax.set_xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('DistilBERT Word Embeddings — PCA Projection to 2D', fontsize=13)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Context Changes Embeddings — Polysemy
DistilBERT produces **contextual** embeddings, unlike static Word2Vec.
The word *bank* has different meanings depending on context.

In [ ]:
def get_word_in_context_embedding(sentence: str, target_word: str) -> np.ndarray:
    """Extract the embedding for a specific token within a sentence."""
    tokens = tokenizer(sentence, return_tensors='pt')
    token_strings = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])

    with torch.no_grad():
        out = model(**tokens)
    hidden = out.last_hidden_state[0]  # (seq_len, 768)

    # Find which token index corresponds to the target word (first match)
    target_lower = target_word.lower()
    for i, tok in enumerate(token_strings):
        if target_lower in tok.replace('#', '').lower():
            return hidden[i].numpy(), i, token_strings
    # Fallback: mean pool
    return hidden.mean(0).numpy(), -1, token_strings


bank_sentences = [
    ("I went to the bank to deposit my paycheck.",          "bank"),
    ("The fishermen sat on the river bank all morning.",    "bank"),
    ("The bank approved my mortgage application.",          "bank"),
    ("We had to bank the aircraft to avoid turbulence.",    "bank"),
]

bank_embs  = []
bank_labels = []
for sent, word in bank_sentences:
    emb, idx, toks = get_word_in_context_embedding(sent, word)
    bank_embs.append(emb)
    bank_labels.append(sent[:40] + '...')

bank_matrix = np.stack(bank_embs)

# Cosine sims between all pairs
norms = np.linalg.norm(bank_matrix, axis=1, keepdims=True)
normed = bank_matrix / norms
sim = normed @ normed.T

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(sim, xticklabels=bank_labels, yticklabels=bank_labels,
            annot=True, fmt='.3f', cmap='Blues', vmin=0.8, vmax=1.0,
            linewidths=1, ax=ax)
ax.set_title('Contextual Similarity of "bank" in Different Sentences\n(DistilBERT — contextual embeddings)', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()
print('\nObserve: financial bank sentences cluster together; river/aircraft differ.')

## 5. Semantic Search — Nearest Neighbors
Given a query sentence, find the most semantically similar sentence from a corpus using cosine similarity.

In [ ]:
corpus = [
    "Transformers use self-attention to process sequences in parallel.",
    "Deep neural networks have many layers of computation.",
    "The stock market experienced significant volatility yesterday.",
    "Gradient descent optimizes model weights during training.",
    "Natural language processing enables computers to understand text.",
    "Interest rates were raised by the central bank this quarter.",
    "Backpropagation computes gradients for each parameter.",
    "Machine learning models learn patterns from data.",
    "The attention mechanism allows focusing on relevant parts of the input.",
    "Inflation affects purchasing power and consumer spending.",
]

print('Encoding corpus...')
corpus_embs = np.stack([get_embedding(s, 'mean') for s in corpus])
corpus_norms = np.linalg.norm(corpus_embs, axis=1, keepdims=True)
corpus_normed = corpus_embs / corpus_norms

def semantic_search(query: str, top_k: int = 3):
    q_emb = get_embedding(query, 'mean')
    q_norm = q_emb / np.linalg.norm(q_emb)
    sims = corpus_normed @ q_norm
    ranked = np.argsort(sims)[::-1]
    print(f'\nQuery: "{query}"')
    print('Top matches:')
    for rank, idx in enumerate(ranked[:top_k], 1):
        bar = '█' * int(sims[idx] * 30)
        print(f'  {rank}. [{sims[idx]:.3f}] {bar}')
        print(f'     {corpus[idx]}')

semantic_search('How does the neural network learn?')
semantic_search('What controls how the model attends to tokens?')
semantic_search('What is happening with the economy?')

## Summary
- Embeddings map discrete tokens into a continuous, high-dimensional vector space (768 dims for DistilBERT).
- Semantically related words cluster together — measurable via cosine similarity.
- DistilBERT produces **contextual** embeddings: the same word gets different vectors depending on context.
- PCA lets us visualize high-dimensional clusters in 2D, revealing semantic structure.
- Semantic search uses embedding cosine similarity to find the most relevant documents.

> **Next notebook:** Attention Weights — visualize which tokens the model attends to.